# RFQ pipeline - homeowner journey on Vertex AI

Runs the full customer journey on Google Cloud: postcode in, installer-ready RFQ out.

Nothing here reimplements the pipeline. Every cell calls the same functions the live API
calls, so what you see is production behaviour, not a notebook-only reproduction.

**No GPU required.** Generation runs on Vertex AI Model-as-a-Service (Llama 3.3 70B), so
this instance only orchestrates. The cheapest CPU machine is the right choice.

Setup instructions are in `notebooks/README.md`.

## 1. Setup

In [1]:
import sys, json, os
from pathlib import Path

# Repo root, so the pipeline modules import from a notebook subdirectory.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

def show(obj, title=None):
    if title:
        print(f"\n=== {title} ===")
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

print("repo root :", ROOT)
print("EPC token :", "set" if os.getenv("EPC_BEARER_TOKEN") else "MISSING")
print("project   :", os.getenv("GOOGLE_CLOUD_PROJECT") or "MISSING")
print("LLM model :", os.getenv("LLM_MODEL") or "(default from generate_rfq.MODEL_NAME)")
print("SA key    :", os.getenv("GOOGLE_APPLICATION_CREDENTIALS") or "unset (correct on Workbench: ADC uses the instance identity)")

repo root : /Users/vanessrw/Desktop/renbee/main
EPC token : set
project   : renbee-500415
LLM model : meta/llama-3.3-70b-instruct-maas
SA key    : /Users/vanessrw/Desktop/renbee_july/ThesisNessy/renbee-500415-3f335a9b7b5b.json


## 2. Step 1 — what the homeowner enters

The only two things asked upfront. Edit these to try a different journey.

| Postcode | Exercises |
|---|---|
| `E1 6AN` | normal EPC + Bishopsgate conservation area |
| `BA1 1LZ` | no EPC at the postcode, nearby-property picker |
| `SW1A 2AA` | no EPC anywhere, manual fallback |

In [2]:
POSTCODE   = "E1 6AN"
TECHNOLOGY = "heat_pump"   # heat_pump | solar_pv | battery | solar_thermal

## 3. Look up the property's EPC

In [3]:
from epc_fetch import fetch_epc_data

epc_data = fetch_epc_data(POSTCODE)
print(f"\n{epc_data['count']} certificate(s) at {POSTCODE}")
for p in epc_data["properties"][:5]:
    c = p["certificate"]
    print(f"  {c.get('address')} — rating {c.get('current-energy-rating')}")

Searching EPC records for postcode: E1 6AN


/Users/vanessrw/Desktop/renbee/main/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Found 3 properties

  6, Brushfield Street — Rating: D
  6, Brushfield Street — Rating: D
  6 Brushfield Street — Rating: D

Saved to /Users/vanessrw/Desktop/renbee/main/output/E1_6AN.json

3 certificate(s) at E1 6AN
  6, Brushfield Street — rating D
  6, Brushfield Street — rating D
  6 Brushfield Street — rating D


## 4. Address resolution

One address resolves silently. Several, and the homeowner is shown a picker. This is the
`409 ambiguous_address` branch of the live API.

In [4]:
from epc_to_rfq import AmbiguousAddress, select_certificate

chosen_lmk = None
try:
    chosen = select_certificate(epc_data)
    print("Resolved to a single address:" if chosen else "No EPC found, manual entry path.")
    if chosen:
        print(" ", chosen["certificate"].get("address"))
except AmbiguousAddress as e:
    print(f"Homeowner would pick from {len(e.candidates)} addresses:")
    for c in e.candidates[:8]:
        print(f"  [{c['lmk_key']}] {c['address']}  ({c['inspection_date']})")
    chosen_lmk = e.candidates[0]["lmk_key"]   # the notebook picks the first
    print(f"\nPicking: {e.candidates[0]['address']}")

Homeowner would pick from 2 addresses:
  [9352-2833-6699-0908-7615] 6 Brushfield Street  (2008-11-07)
  [0352-2837-6649-9921-7631] 6, Brushfield Street  (2019-04-23)

Picking: 6 Brushfield Street


## 5. Site intelligence

Cross-references the postcode against planning.data.gov.uk. The homeowner never asks for
this, it is checked for them.

In [5]:
from epc_to_rfq import build_site_context

site_context = build_site_context(POSTCODE)
planning = site_context.get("planning") or {}
hits = {k: v for k, v in planning.items() if v}
show(hits or {"note": "no planning constraints recorded"}, "Planning constraints")
print("\nSources:", site_context.get("data_sources"))


=== Planning constraints ===
{
  "conservation_area_name": "Bishopsgate",
  "permitted_development_restricted": true,
  "planning_permission_likely_required": true,
  "consent_may_be_required": true,
  "planning_consequence_basis": "conservation area: Bishopsgate"
}

Sources: ['planning.data.gov.uk']


## 6. Assemble, and see the form the homeowner gets

Everything the EPC supplied is auto-filled. `missing_fields` is what still has to be
asked, `optional_fields` is offered but never blocks.

In [6]:
from epc_to_rfq import assemble_rfq_input, missing_fields, optional_fields, completeness_score

rfq_input = assemble_rfq_input(
    epc_data,
    {"common": {"technology_requested": TECHNOLOGY}},
    lmk_key=chosen_lmk,
)
rfq_input["site_context"] = site_context

show(rfq_input["property"], "Auto-filled from the EPC")

print("\n=== The form the homeowner is shown ===")
for section, fields in missing_fields(rfq_input).items():
    print(f"\n[{section}] required")
    for f in fields:
        print(f"  - {f['label']}")
for section, fields in optional_fields(rfq_input).items():
    print(f"\n[{section}] optional")
    for f in fields:
        print(f"  - {f['label']}")

c = completeness_score(rfq_input)
print(f"\nCompleteness before the form: {c['populated']}/{c['required']} = {c['score']:.2f}")


=== Auto-filled from the EPC ===
{
  "epc_found": true,
  "property_type": "maisonette",
  "built_form": "end-terrace",
  "floor_area_m2": null,
  "construction_age_band": "1900-1929",
  "epc_rating": "D",
  "epc_score": 57,
  "current_heating_system": "Boiler and radiators, mains gas",
  "current_fuel_type": null,
  "hot_water_system": "From main system",
  "walls_description": "Solid brick, as built, no insulation (assumed)",
  "roof_description": "Roof room(s), no insulation (assumed)",
  "windows_description": null,
  "occupancy_status": null,
  "access_constraints": null
}

=== The form the homeowner is shown ===

[common] required
  - How would you prefer to be contacted?
  - Your email address
  - Your phone number
  - When would you like the installation?
  - What is your main reason for this enquiry?

[property] required
  - Current fuel type

[heat_pump] required
  - Which kind of heat pump?
  - What kind of heat emitters do you have?
  - Is there space indoors for a hot wat

## 7. The human step

What the homeowner types. Adjust to match whichever technology you chose above, using the
field names printed by the previous cell.

In [7]:
answers = {
    "common": {
        "preferred_contact_method": "email",
        "contact_email": "homeowner@example.com",
        "contact_phone": "07700 900123",
        "desired_installation_timeline": "within_6_months",
        "motivation": "Cut gas use and lower our heating bills.",
    },
    "heat_pump": {
        "heat_pump_type_interest": "ground_air_source",
        "emitter_type": "radiators",
        "hot_water_cylinder_space_available": "yes",
        "external_unit_space": "yes",
        "garden_or_side_access": "yes",
        "number_of_bedrooms": 3,
        "number_of_bathrooms": 2,
        "number_of_occupants": 4,
        "smart_meter_installed": "yes",
        "smart_meter_cutout_fuse_label": "100A",
    },
}

for section, fields in answers.items():
    rfq_input.setdefault(section, {}).update(fields)

still_missing = missing_fields(rfq_input)
c = completeness_score(rfq_input)
print(f"Completeness now: {c['populated']}/{c['required']} = {c['score']:.2f}")
print("Still blocking:", still_missing or "nothing — ready to generate")

Completeness now: 17/18 = 0.94
Still blocking: {'property': [{'name': 'current_fuel_type', 'label': 'Current fuel type', 'type': 'select', 'options': ['mains_gas', 'heating_oil', 'lpg', 'electricity', 'biomass_or_wood', 'coal_or_solid_fuel', 'district_heating', 'other_or_unsure']}]}


## 8. LLM moment 1 — the homeowner-facing recommendation

First live Vertex call. Explains the EPC findings in plain English, with costs and
savings taken verbatim from the certificate.

In [8]:
from generate_rfq import generate_recommendation, MODEL_NAME

print(f"Generating via {os.getenv('LLM_MODEL') or MODEL_NAME} ...\n")
rec = generate_recommendation(rfq_input)
print(rec["recommendation_summary"])
print("\n--", rec["recommendation_disclaimer"])
print("\nparse_status:", rec["parse_status"])

Generating via meta/llama-3.3-70b-instruct-maas ...

Your property has recommendation items on record. [mock generator]

-- Based only on official EPC recommendation data.

parse_status: ok


## 9. LLM moment 2 — the installer-facing RFQ

Second live Vertex call, a different prompt and a different audience. Contact details are
stripped before the model sees them.

In [9]:
from generate_rfq import generate_rfq_summary

out = generate_rfq_summary(rfq_input)
print(out["rfq_summary"])
print("\nparse_status:", out["parse_status"])

summary = out["rfq_summary"]
print("\nContact details leaked into the prose?",
      any(x in summary for x in ["homeowner@example.com", "900123"]))

The homeowner is requesting a quotation based on property details provided. [mock generator]

parse_status: ok

Contact details leaked into the prose? False


## 10. Human-in-the-loop review

What a reviewer edits before it reaches the installer. Contact details are absent from
the prose above but present here, because the installer needs them.

In [10]:
show(rfq_input, "Final rfq_input (installer receives this verbatim)")


=== Final rfq_input (installer receives this verbatim) ===
{
  "common": {
    "technology_requested": "heat_pump",
    "enquiry_id": "RFQ_EA520C2D",
    "submission_date": "2026-08-31",
    "postcode": "E1 6AN",
    "preferred_contact_method": "email",
    "contact_email": "homeowner@example.com",
    "contact_phone": "07700 900123",
    "desired_installation_timeline": "within_6_months",
    "motivation": "Cut gas use and lower our heating bills."
  },
  "property": {
    "epc_found": true,
    "property_type": "maisonette",
    "built_form": "end-terrace",
    "floor_area_m2": null,
    "construction_age_band": "1900-1929",
    "epc_rating": "D",
    "epc_score": 57,
    "current_heating_system": "Boiler and radiators, mains gas",
    "current_fuel_type": null,
    "hot_water_system": "From main system",
    "walls_description": "Solid brick, as built, no insulation (assumed)",
    "roof_description": "Roof room(s), no insulation (assumed)",
    "windows_description": null,
    "oc

## 11. The same journey over HTTP

The cells above call the library directly. This one starts the real FastAPI service
inside this instance and calls it over HTTP, which is what proves the deployed service
runs on Google Cloud rather than just the functions.

The demo UI is then reachable through the Workbench proxy — see `notebooks/README.md`.

In [11]:
# Standalone-safe: re-assert the repo root and inputs so this cell works
# even if the earlier cells have not been run in this kernel.
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
POSTCODE = globals().get("POSTCODE", "E1 6AN")
TECHNOLOGY = globals().get("TECHNOLOGY", "heat_pump")

import threading, time, requests, uvicorn
import app as app_module

config = uvicorn.Config(app_module.app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

print("GET /health ->", requests.get("http://127.0.0.1:8000/health", timeout=30).json())

r = requests.post(
    "http://127.0.0.1:8000/api/initiate",
    json={"postcode": POSTCODE, "technology": TECHNOLOGY},
    timeout=180,
)
print("\nPOST /api/initiate ->", r.status_code)
body = r.json()
if r.status_code == 200:
    print("session:", body["session_id"])
    print("epc_found:", body["epc_found"])
    print("required still to ask:", {k: [f["name"] for f in v] for k, v in body["missing_fields"].items()})
else:
    print("branch:", body.get("detail", {}).get("error"))

ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


GET /health -> {'status': 'ok', 'demo_mock_llm': True, 'llm_target': 'meta/llama-3.3-70b-instruct-maas @ https://us-central1-aiplatform.googleapis.com/v1beta1/projects/renbee-500415/locations/us-central1/endpoints/openapi'}

POST /api/initiate -> 409
branch: ambiguous_address
